# ModeSwitch-LLM Selector Notebook

This notebook is the selector side of the project.

The point here is not to rerun anything. It is to take the benchmark measurements that already exist in the repo, turn them into a small routing problem, and check whether a lightweight selector is actually justified.

All quantitative tables below come from parsing the saved outputs in:

- `baseline_smoke_test (3) (1).ipynb`
- `stress_test (2).ipynb`

`stress_test_plots.ipynb` and the saved report screenshots only help with interpretation. They do not add new measurements.


## Proposal Alignment

The proposal asks for four things on the selector side:

1. static baselines
2. a phase-agnostic comparison baseline
3. a simple phase-aware selector
4. routing based on simple runtime signals like prompt length, expected output length, memory pressure, and latency target

So the notebook is set up to do exactly that. It keeps the routing logic simple, it does not invent any new runs, and it keeps the final claim narrow enough to match the actual evidence.


In [ ]:
from __future__ import annotations

import json
import re
from io import StringIO
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CONTROLLER_NOTEBOOK = ROOT / "baseline_smoke_test (3) (1).ipynb"
STRESS_NOTEBOOK = ROOT / "stress_test (2).ipynb"
CONFIG_PATH = ROOT / "config.py"

RUN_RE = re.compile(
    r"\[(\d+)/(\d+)\]\s+([A-Za-z0-9_]+)\s+\|\s+([A-Za-z0-9_]+)\s+\|\s+trial=(\d+)\s+\| "
    r"ttft=([\d.]+) ms \| lat=([\d.]+) ms \| tps=([\d.]+) \| J/tok=([\d.]+) \| gpu=([\d.]+) MB"
)
BASE_MODEL_RE = re.compile(r'^\s*model_name_or_path:\s*str\s*=\s*"([^"]+)"', re.M)
TOKENIZER_RE = re.compile(r'^\s*tokenizer_name_or_path:\s*Optional\[str\]\s*=\s*"([^"]+)"', re.M)
SPECIAL_MODEL_RE = re.compile(
    r'^\s*(\w+_model_name_or_path):\s*Optional\[str\]\s*=\s*"([^"]+)"', re.M
)
OUTPUT_TOKENS = {
    "ctx4032_out64": 64,
    "ctx3840_out256": 256,
    "shared4032_out64": 64,
    "ctx2048_out64": 64,
}
STATIC_BASELINE_EXPECTED = {
    "static_fp16": {
        "mean_ttft_ms": 279.397,
        "mean_total_latency_ms": 3716.281,
        "mean_tokens_per_second": 41.498,
        "mean_energy_per_token_j": 4.947,
        "mean_quality_score": 1.0,
    },
    "static_int8": {
        "mean_ttft_ms": 224.595,
        "mean_total_latency_ms": 2009.324,
        "mean_tokens_per_second": 75.931,
        "mean_energy_per_token_j": 2.616,
        "mean_quality_score": 0.83682,
    },
    "static_speculative": {
        "mean_ttft_ms": 353.023,
        "mean_total_latency_ms": 1499.481,
        "mean_tokens_per_second": 98.161,
        "mean_energy_per_token_j": 2.382,
        "mean_quality_score": 1.0,
    },
    "static_gptq": {
        "mean_ttft_ms": 319.298,
        "mean_total_latency_ms": 1625.135,
        "mean_tokens_per_second": 92.843,
        "mean_energy_per_token_j": 2.385,
        "mean_quality_score": 0.228814,
    },
}
QUALITY_THRESHOLD = 0.8


def parse_logged_runs(path: Path) -> pd.DataFrame:
    nb = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for cell in nb["cells"]:
        for output in cell.get("outputs", []):
            if output.get("output_type") != "stream":
                continue
            text = "".join(output.get("text", []))
            for match in RUN_RE.finditer(text):
                workload_name = match.group(4)
                ttft_ms = float(match.group(6))
                total_latency_ms = float(match.group(7))
                output_tokens = OUTPUT_TOKENS.get(workload_name)
                rows.append(
                    {
                        "mode_name": match.group(3),
                        "workload_name": workload_name,
                        "trial_index": int(match.group(5)),
                        "ttft_ms": ttft_ms,
                        "total_latency_ms": total_latency_ms,
                        "tokens_per_second": float(match.group(8)),
                        "energy_per_token_j": float(match.group(9)),
                        "peak_gpu_memory_mb": float(match.group(10)),
                        "output_tokens": output_tokens,
                        "prefill_ms": ttft_ms,
                        "tbt_ms": ((total_latency_ms - ttft_ms) / output_tokens)
                        if output_tokens
                        else None,
                    }
                )
    return pd.DataFrame(rows)


def aggregate_runs(df: pd.DataFrame, source: str) -> pd.DataFrame:
    metrics = [
        "ttft_ms",
        "total_latency_ms",
        "tokens_per_second",
        "energy_per_token_j",
        "peak_gpu_memory_mb",
        "prefill_ms",
        "tbt_ms",
    ]
    out = df.groupby(["mode_name", "workload_name"])[metrics].agg(["mean", "std"]).reset_index()
    out.columns = ["mode_name", "workload_name"] + [
        f"{metric}_{stat}" for metric, stat in out.columns.tolist()[2:]
    ]
    out["source"] = source
    return out


def load_html_table(path: Path, cell_index: int, output_index: int) -> pd.DataFrame:
    nb = json.loads(path.read_text(encoding="utf-8"))
    output = nb["cells"][cell_index]["outputs"][output_index]
    html = output["data"]["text/html"]
    if isinstance(html, list):
        html = "".join(html)
    return pd.read_html(StringIO(html))[0]


def extract_model_inventory(config_path: Path) -> pd.DataFrame:
    config_text = config_path.read_text(encoding="utf-8")
    base_model = BASE_MODEL_RE.search(config_text).group(1)
    tokenizer = TOKENIZER_RE.search(config_text).group(1)
    rows = [
        {
            "mode_family": "base_fp16",
            "deployed_model": base_model,
            "notes": "Main checkpoint used for fp16_baseline and non-quantized vLLM modes.",
        },
        {
            "mode_family": "tokenizer",
            "deployed_model": tokenizer,
            "notes": "Tokenizer paired with the base checkpoint.",
        },
    ]
    for key, value in SPECIAL_MODEL_RE.findall(config_text):
        rows.append(
            {
                "mode_family": key.replace("_model_name_or_path", ""),
                "deployed_model": value,
                "notes": "Configured special checkpoint or draft model.",
            }
        )
    rows.extend(
        [
            {
                "mode_family": "runtime variants",
                "deployed_model": base_model,
                "notes": "prefix_caching, chunked_prefill, continuous_batching, and cuda_graphs all sit on the base model.",
            },
            {
                "mode_family": "hybrid modes",
                "deployed_model": "compositions of configured single modes",
                "notes": "Measured directly in the long-context stress notebook.",
            },
        ]
    )
    return pd.DataFrame(rows)


def build_quality_proxy(controller_agg: pd.DataFrame) -> dict[str, float]:
    mode_quality = (
        controller_agg.groupby("mode_name", as_index=False)["quality_score"]
        .median()
        .sort_values("quality_score", ascending=False)
    )
    base_quality = dict(zip(mode_quality["mode_name"], mode_quality["quality_score"]))
    for hybrid_name, components in {
        "int8_plus_chunked_prefill": ["int8_quant", "chunked_prefill"],
        "gptq_plus_chunked_prefill": ["gptq_4bit", "chunked_prefill"],
        "speculative_plus_chunked_prefill": ["speculative_decoding", "chunked_prefill"],
        "int8_plus_prefix_caching": ["int8_quant", "prefix_caching"],
        "gptq_plus_prefix_caching": ["gptq_4bit", "prefix_caching"],
        "int8_plus_continuous_batching": ["int8_quant", "continuous_batching"],
        "gptq_plus_continuous_batching": ["gptq_4bit", "continuous_batching"],
    }.items():
        base_quality[hybrid_name] = min(base_quality[component] for component in components)
    return base_quality


def route(selector_name: str, signals_row: pd.Series) -> str:
    if selector_name == "static_fp16":
        return "fp16_baseline"
    if selector_name == "static_int8":
        return "int8_quant"
    if selector_name == "static_speculative":
        return "speculative_decoding"
    if selector_name == "static_gptq":
        return "gptq_4bit"

    if selector_name == "phase_agnostic_simple":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return "prefix_caching"
        if signals_row["predicted_output_length"] <= 64:
            return "speculative_decoding"
        return "int8_quant"

    if selector_name == "phase_aware_quality":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return (
                "int8_plus_prefix_caching"
                if signals_row["memory_pressure"] >= 0.9
                else "prefix_caching"
            )
        if (
            signals_row["predicted_output_length"] <= 64
            and signals_row["latency_target"] == "interactive"
            and signals_row["memory_pressure"] < 0.98
        ):
            return "speculative_decoding"
        return "int8_quant"

    if selector_name == "phase_aware_aggressive":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return "gptq_plus_prefix_caching"
        if signals_row["predicted_output_length"] <= 64:
            return "gptq_4bit"
        return "speculative_decoding"

    raise KeyError(f"Unknown selector: {selector_name}")


def build_selector_routes(workload_signals_df: pd.DataFrame, selector_names: list[str]) -> pd.DataFrame:
    core_signals = workload_signals_df[workload_signals_df["workload_name"] != "ctx2048_out64"].copy()
    rows = []
    for selector_name in selector_names:
        for _, signal_row in core_signals.iterrows():
            row_dict = signal_row.to_dict()
            rows.append(
                {
                    **row_dict,
                    "selector": selector_name,
                    "chosen_mode": route(selector_name, signal_row),
                }
            )
    return pd.DataFrame(rows)


def summarize_selector_rows(df: pd.DataFrame) -> pd.DataFrame:
    usable = df[df["measured"]].copy()
    summary = (
        usable.groupby("selector", as_index=False)
        .agg(
            workloads_covered=("workload_name", "nunique"),
            mean_ttft_ms=("ttft_ms", "mean"),
            mean_prefill_ms=("prefill_ms", "mean"),
            mean_tbt_ms=("tbt_ms", "mean"),
            mean_total_latency_ms=("total_latency_ms", "mean"),
            mean_tokens_per_second=("tokens_per_second", "mean"),
            mean_energy_per_token_j=("energy_per_token_j", "mean"),
            mean_quality_score=("quality_score", "mean"),
            mean_ttft_ms_std=("ttft_ms_std", "mean"),
            mean_prefill_ms_std=("prefill_ms_std", "mean"),
            mean_tbt_ms_std=("tbt_ms_std", "mean"),
            mean_total_latency_ms_std=("total_latency_ms_std", "mean"),
            mean_tokens_per_second_std=("tokens_per_second_std", "mean"),
            mean_energy_per_token_j_std=("energy_per_token_j_std", "mean"),
        )
        .sort_values("mean_total_latency_ms")
        .reset_index(drop=True)
    )
    summary["passes_quality_threshold"] = summary["mean_quality_score"] >= QUALITY_THRESHOLD
    return summary


def sanity_check_static_baselines(summary_df: pd.DataFrame, tolerance: float = 0.02) -> None:
    for selector_name, expected_values in STATIC_BASELINE_EXPECTED.items():
        row = summary_df.loc[summary_df["selector"] == selector_name]
        if row.empty:
            raise AssertionError(f"Missing selector row for {selector_name}.")
        row = row.iloc[0]
        for column_name, expected_value in expected_values.items():
            actual_value = float(row[column_name])
            if abs(actual_value - expected_value) > tolerance:
                raise AssertionError(
                    f"{selector_name} {column_name} mismatch: got {actual_value}, expected {expected_value}"
                )


In [ ]:
controller_runs = parse_logged_runs(CONTROLLER_NOTEBOOK)
stress_runs = parse_logged_runs(STRESS_NOTEBOOK)

controller_agg = aggregate_runs(controller_runs, "controller_benchmark")
stress_agg = aggregate_runs(stress_runs, "stress_benchmark")

controller_agg_html = load_html_table(CONTROLLER_NOTEBOOK, 4, 1).drop(
    columns=["Unnamed: 0", "..."], errors="ignore"
)
controller_cmp = load_html_table(CONTROLLER_NOTEBOOK, 4, 3).drop(
    columns=["Unnamed: 0", "..."], errors="ignore"
)

controller_quality = controller_agg_html[
    ["mode_name", "workload_name", "baseline_similarity_rouge_l_f1_mean"]
].rename(columns={"baseline_similarity_rouge_l_f1_mean": "quality_score"})
controller_agg = controller_agg.merge(
    controller_quality, on=["mode_name", "workload_name"], how="left"
)

quality_proxy = build_quality_proxy(controller_agg)
stress_agg["quality_score"] = stress_agg["mode_name"].map(quality_proxy)
all_agg = pd.concat([controller_agg, stress_agg], ignore_index=True, sort=False)

print(f"Controller runs parsed: {len(controller_runs)}")
print(f"Stress and hybrid runs parsed: {len(stress_runs)}")
print(f"Controller mode-workload pairs: {controller_agg.shape[0]}")
print(f"Stress mode-workload pairs: {stress_agg.shape[0]}")


## Data Sources

The selector analysis is built from two measurement sources and one interpretation source:

- the controller sweep for broad fixed-mode behavior
- the long-context stress notebook for long-context and hybrid behavior
- the plot/report material for extra controller cues only


In [ ]:
benchmark_inventory_df = pd.DataFrame(
    [
        {
            "artifact": "baseline_smoke_test (3) (1).ipynb",
            "role": "broad controller sweep",
            "parsed_runs": int(len(controller_runs)),
            "mode_workload_pairs": int(controller_agg.shape[0]),
            "what_it_adds": "Fixed-mode behavior across short, long, memory-pressure, and shared-prefix cases.",
        },
        {
            "artifact": "stress_test (2).ipynb",
            "role": "long-context and hybrid sweep",
            "parsed_runs": int(len(stress_runs)),
            "mode_workload_pairs": int(stress_agg.shape[0]),
            "what_it_adds": "Measured long-context hybrids like prefix caching, chunked prefill, and continuous batching.",
        },
        {
            "artifact": "stress_test_plots.ipynb",
            "role": "interpretation only",
            "parsed_runs": 0,
            "mode_workload_pairs": 0,
            "what_it_adds": "Summarizes what the stress benchmark implies for controller logic.",
        },
    ]
)
benchmark_inventory_df


In [ ]:
model_inventory_df = extract_model_inventory(CONFIG_PATH)
model_inventory_df


## Fixed-Mode Baselines

Before looking at selector logic, it helps to know which single modes are even worth keeping in the first controller version.

The next two tables come from the controller sweep:

- a compact fixed-mode ranking
- the quality-safe winner per workload using the same `quality_score >= 0.8` rule that will be used later for selectors


In [ ]:
controller_mode_summary = (
    controller_cmp.groupby("mode_name", as_index=False)
    .agg(
        workloads_measured=("workload_name", "count"),
        mean_latency_speedup=("latency_speedup_vs_baseline", "mean"),
        mean_throughput_ratio=("throughput_ratio_vs_baseline", "mean"),
        mean_energy_ratio=("energy_per_token_ratio_vs_baseline", "mean"),
        median_quality=("baseline_similarity_rouge_l_f1_mean", "median"),
    )
    .sort_values(["median_quality", "mean_latency_speedup"], ascending=[False, False])
    .reset_index(drop=True)
)
controller_mode_summary["takeaway"] = [
    "strong quality-safe decode specialist" if name == "speculative_decoding"
    else "best general fixed-mode baseline" if name == "int8_quant"
    else "strong repeated-prefix fixed-mode baseline" if name == "prefix_caching"
    else "useful phase primitive more than a default whole-request mode" if name == "chunked_prefill"
    else "better as a serving extension than a default interactive branch" if name == "continuous_batching"
    else "quality-safe but not strong enough to prioritize first" if name == "cuda_graphs"
    else "not strong enough for the first selector version" if name == "kv_cache_compression"
    else "fast but quality-limited" if ("gptq" in name or "awq" in name)
    else "measured fixed mode"
    for name in controller_mode_summary["mode_name"]
]
controller_mode_summary


In [ ]:
quality_safe_best = (
    controller_cmp[controller_cmp["baseline_similarity_rouge_l_f1_mean"] >= QUALITY_THRESHOLD]
    .groupby("workload_name")["latency_speedup_vs_baseline"]
    .idxmax()
)
quality_safe_best = (
    controller_cmp.loc[
        quality_safe_best,
        [
            "workload_name",
            "mode_name",
            "latency_speedup_vs_baseline",
            "energy_per_token_ratio_vs_baseline",
            "baseline_similarity_rouge_l_f1_mean",
        ],
    ]
    .sort_values("workload_name")
    .reset_index(drop=True)
    .rename(
        columns={
            "mode_name": "quality_safe_winner",
            "latency_speedup_vs_baseline": "latency_speedup_vs_fp16",
            "energy_per_token_ratio_vs_baseline": "energy_ratio_vs_fp16",
            "baseline_similarity_rouge_l_f1_mean": "quality_score",
        }
    )
)
quality_safe_best


## Extra Controller Cues From the Plot and Report Material

The plot notebook and the saved screenshots do not add new runs, but they do help explain how the selector should be read.

I only use them here for qualitative guidance:

- decode dominates most cases, so decode-efficient modes matter a lot
- repeated-prefix traffic is where the strongest phase-aware branch shows up
- batching hybrids look real, but they fit better as a serving extension than as the default single-request controller


In [ ]:
report_cues_df = pd.DataFrame(
    [
        {
            "source": "Saved report screenshot",
            "observation": "int8_plus_continuous_batching is the most energy-efficient mode on several decode-heavy workloads.",
            "selector_implication": "Keep batching as a serving extension, not as the default interactive route.",
        },
        {
            "source": "Saved report screenshot",
            "observation": "gptq_plus_prefix_caching is the most energy-efficient repeated-prefix hybrid.",
            "selector_implication": "Repeated-prefix traffic is the clearest place where phase-aware logic matters.",
        },
        {
            "source": "Saved phase-dominance screenshot",
            "observation": "Most workloads are decode-dominated, but some long-context cases still have noticeable prefill cost.",
            "selector_implication": "The notebook should report both prefill and token-by-token decode behavior instead of total latency alone.",
        },
        {
            "source": "stress_test_plots.ipynb",
            "observation": "Speculative decoding wins 3840-in / 256-out among single modes, while gptq_4bit wins 4032-in / 64-out by raw latency.",
            "selector_implication": "Use speculative as the quality-safe decode branch and keep GPTQ inside the aggressive ablation only.",
        },
    ]
)
report_cues_df


## Runtime Signals

The proposal wanted simple runtime signals, not a learned controller.

So the routing logic below uses:

- prompt length
- predicted output length
- repeated-prefix structure
- memory pressure
- latency target

`memory_pressure` and `latency_target` are both active in the routing:

- `memory_pressure` decides whether repeated-prefix traffic stays on plain `prefix_caching` or moves to `int8_plus_prefix_caching`
- `latency_target` is what moves the serving extension row toward `int8_plus_continuous_batching`


In [ ]:
workload_signals_df = pd.DataFrame(
    [
        {
            "workload_name": "ctx4032_out64",
            "prompt_length": 4032,
            "predicted_output_length": 64,
            "repeated_prefix": False,
            "memory_pressure": 0.95,
            "latency_target": "interactive",
            "quality_priority": "high",
        },
        {
            "workload_name": "ctx3840_out256",
            "prompt_length": 3840,
            "predicted_output_length": 256,
            "repeated_prefix": False,
            "memory_pressure": 0.95,
            "latency_target": "balanced",
            "quality_priority": "high",
        },
        {
            "workload_name": "shared4032_out64",
            "prompt_length": 3936,
            "predicted_output_length": 64,
            "repeated_prefix": True,
            "memory_pressure": 0.92,
            "latency_target": "interactive",
            "quality_priority": "high",
        },
        {
            "workload_name": "ctx2048_out64",
            "prompt_length": 2048,
            "predicted_output_length": 64,
            "repeated_prefix": False,
            "memory_pressure": 0.55,
            "latency_target": "throughput_serving",
            "quality_priority": "medium",
        },
    ]
)
workload_signals_df


## Measured Candidate Table

This is the actual measured candidate pool for the core selector comparison.

Two details matter here:

- `prefill_ms = ttft_ms`
- `tbt_ms = (total_latency_ms - ttft_ms) / output_tokens`

That second metric is important because the whole project is built around the prefill vs decode split.

For hybrid quality, the notebook keeps the conservative proxy used before:

- hybrid quality = minimum quality of the component single modes

That is cautious on purpose. It avoids claiming a hybrid is quality-safe when the repo never measured hybrid ROUGE directly.


In [ ]:
selector_cases = (
    all_agg[all_agg["workload_name"].isin(["ctx4032_out64", "ctx3840_out256", "shared4032_out64"])]
    .dropna(subset=["quality_score"])
    .copy()
)
selector_cases = selector_cases.rename(
    columns={
        "ttft_ms_mean": "ttft_ms",
        "ttft_ms_std": "ttft_ms_std",
        "prefill_ms_mean": "prefill_ms",
        "prefill_ms_std": "prefill_ms_std",
        "tbt_ms_mean": "tbt_ms",
        "tbt_ms_std": "tbt_ms_std",
        "total_latency_ms_mean": "total_latency_ms",
        "total_latency_ms_std": "total_latency_ms_std",
        "tokens_per_second_mean": "tokens_per_second",
        "tokens_per_second_std": "tokens_per_second_std",
        "energy_per_token_j_mean": "energy_per_token_j",
        "energy_per_token_j_std": "energy_per_token_j_std",
        "peak_gpu_memory_mb_mean": "peak_gpu_memory_mb",
        "peak_gpu_memory_mb_std": "peak_gpu_memory_mb_std",
    }
)
measured_candidate_df = selector_cases[
    [
        "workload_name",
        "mode_name",
        "source",
        "ttft_ms",
        "ttft_ms_std",
        "prefill_ms",
        "prefill_ms_std",
        "tbt_ms",
        "tbt_ms_std",
        "total_latency_ms",
        "total_latency_ms_std",
        "tokens_per_second",
        "tokens_per_second_std",
        "energy_per_token_j",
        "energy_per_token_j_std",
        "quality_score",
    ]
].sort_values(["workload_name", "total_latency_ms", "mode_name"]).reset_index(drop=True)
measured_candidate_df


## Selector Variants

The notebook compares four static baselines and three adaptive policies:

- `static_fp16`
- `static_int8`
- `static_speculative`
- `static_gptq`
- `phase_agnostic_simple`
- `phase_aware_quality`
- `phase_aware_aggressive`

The static baselines are included as anchors.

The three adaptive policies do different jobs:

- `phase_agnostic_simple` is the proposal-required comparison baseline
- `phase_aware_quality` is the conservative final controller candidate
- `phase_aware_aggressive` is the quality-efficiency frontier ablation


In [ ]:
selector_policy_df = pd.DataFrame(
    [
        {
            "selector": "phase_agnostic_simple",
            "routing_rule": "No explicit phase switch. Repeated-prefix goes to prefix_caching, short-output interactive requests go to speculative, and long generations go to int8.",
            "why_it_exists": "Proposal-required phase-agnostic comparison baseline.",
        },
        {
            "selector": "phase_aware_quality",
            "routing_rule": "Same base logic as the simple controller, but repeated-prefix traffic under high memory pressure moves to int8_plus_prefix_caching.",
            "why_it_exists": "Conservative final controller candidate.",
        },
        {
            "selector": "phase_aware_aggressive",
            "routing_rule": "Uses gptq for short-output interactive requests, speculative for long generations, and gptq_plus_prefix_caching on repeated-prefix traffic.",
            "why_it_exists": "Aggressive ablation that trades quality for speed and energy.",
        },
    ]
)
selector_policy_df


In [ ]:
selector_names = [
    "static_fp16",
    "static_int8",
    "static_speculative",
    "static_gptq",
    "phase_agnostic_simple",
    "phase_aware_quality",
    "phase_aware_aggressive",
]

selector_routes_df = build_selector_routes(workload_signals_df, selector_names)

routing_breakdown_df = selector_routes_df.merge(
    measured_candidate_df,
    left_on=["workload_name", "chosen_mode"],
    right_on=["workload_name", "mode_name"],
    how="left",
)
routing_breakdown_df["measured"] = routing_breakdown_df["total_latency_ms"].notna()
routing_breakdown_df = routing_breakdown_df.drop(columns=["mode_name"])

selector_summary_full_df = summarize_selector_rows(routing_breakdown_df)

common_subset_workloads = sorted(
    workload_name
    for workload_name, group in routing_breakdown_df.groupby("workload_name")
    if group["measured"].all()
)
selector_summary_common_subset_df = summarize_selector_rows(
    routing_breakdown_df[routing_breakdown_df["workload_name"].isin(common_subset_workloads)]
)

sanity_check_static_baselines(selector_summary_full_df)

selector_summary_full_df["comparison_scope"] = "full"
selector_summary_common_subset_df["comparison_scope"] = "common_subset"

print("Static baseline sanity checks passed.")
print("Common-subset workloads:", ", ".join(common_subset_workloads))


## Per-Workload Routing Breakdown

This is the table that makes the selector behavior visible.

If the repeated-prefix branch matters, it should show up here directly, not only as a small change inside an average.


In [ ]:
routing_breakdown_df = routing_breakdown_df[
    [
        "workload_name",
        "selector",
        "chosen_mode",
        "prefill_ms",
        "tbt_ms",
        "total_latency_ms",
        "energy_per_token_j",
        "quality_score",
        "measured",
    ]
].sort_values(["selector", "workload_name"]).reset_index(drop=True)
routing_breakdown_df


## Selector Summary Tables

I keep two versions of the summary on purpose:

- `selector_summary_common_subset_df` is the fair apples-to-apples comparison and should be read first
- `selector_summary_full_df` keeps the repeated-prefix row, which is exactly where the phase-aware branch becomes useful

The pass rule is fixed and explicit:

- a selector passes only if `mean_quality_score >= 0.8`


In [ ]:
selector_summary_common_subset_df


In [ ]:
selector_summary_full_df


## Interpretation

The next cell builds the interpretation text directly from the measured tables above so that every number can be traced back to either `measured_candidate_df`, `routing_breakdown_df`, `selector_summary_common_subset_df`, or `selector_summary_full_df`.


In [ ]:
def lookup(summary_df: pd.DataFrame, selector_name: str) -> pd.Series:
    return summary_df.loc[summary_df["selector"] == selector_name].iloc[0]


common_quality = lookup(selector_summary_common_subset_df, "phase_aware_quality")
common_simple = lookup(selector_summary_common_subset_df, "phase_agnostic_simple")
full_quality = lookup(selector_summary_full_df, "phase_aware_quality")
full_aggressive = lookup(selector_summary_full_df, "phase_aware_aggressive")
full_spec = lookup(selector_summary_common_subset_df, "static_speculative")
full_int8 = lookup(selector_summary_common_subset_df, "static_int8")

repeated_prefix_phase_agnostic = routing_breakdown_df[
    (routing_breakdown_df["selector"] == "phase_agnostic_simple")
    & (routing_breakdown_df["workload_name"] == "shared4032_out64")
].iloc[0]
repeated_prefix_phase_aware = routing_breakdown_df[
    (routing_breakdown_df["selector"] == "phase_aware_quality")
    & (routing_breakdown_df["workload_name"] == "shared4032_out64")
].iloc[0]

long_short_spec = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "speculative_decoding")
    & (measured_candidate_df["workload_name"] == "ctx4032_out64")
].iloc[0]
long_short_int8 = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "int8_quant")
    & (measured_candidate_df["workload_name"] == "ctx4032_out64")
].iloc[0]
long_long_spec = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "speculative_decoding")
    & (measured_candidate_df["workload_name"] == "ctx3840_out256")
].iloc[0]
long_long_int8 = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "int8_quant")
    & (measured_candidate_df["workload_name"] == "ctx3840_out256")
].iloc[0]

interpretation_text = f"""On the fair common subset, `phase_aware_quality` and `phase_agnostic_simple` tie exactly at {common_quality['mean_total_latency_ms']:.1f} ms mean total latency and {common_quality['mean_quality_score']:.3f} mean quality because they make the same choices on `ctx4032_out64` and `ctx3840_out256`. See `selector_summary_common_subset_df`.

That tie is actually useful, not a problem. It shows that the phase-aware logic is not getting fake wins from workloads where it does not change anything. The extra gain only appears when the repeated-prefix workload is included in `selector_summary_full_df`.

The clearest phase-aware win is `shared4032_out64` in `routing_breakdown_df`. `phase_agnostic_simple` picks `prefix_caching` and gets {repeated_prefix_phase_agnostic['total_latency_ms']:.1f} ms total latency and {repeated_prefix_phase_agnostic['energy_per_token_j']:.3f} J/token, while `phase_aware_quality` picks `int8_plus_prefix_caching` and gets {repeated_prefix_phase_aware['total_latency_ms']:.1f} ms and {repeated_prefix_phase_aware['energy_per_token_j']:.3f} J/token. The conservative quality proxy for that branch is {repeated_prefix_phase_aware['quality_score']:.3f}. See `routing_breakdown_df`.

The prefill vs decode split also lines up with the controller story in `measured_candidate_df`. On `ctx4032_out64`, `int8_quant` has much better prefill at {long_short_int8['prefill_ms']:.1f} ms than `speculative_decoding` at {long_short_spec['prefill_ms']:.1f} ms, but `speculative_decoding` has much lower token-by-token decode cost at {long_short_spec['tbt_ms']:.3f} ms/token versus {long_short_int8['tbt_ms']:.3f} ms/token, which is why it wins total latency for the short-output interactive case. On `ctx3840_out256`, the same decode effect becomes even stronger: `speculative_decoding` is at {long_long_spec['tbt_ms']:.3f} ms/token while `int8_quant` is at {long_long_int8['tbt_ms']:.3f} ms/token. See `measured_candidate_df`.

Under the explicit pass rule `mean_quality_score >= 0.8`, `phase_aware_quality` passes with {full_quality['mean_quality_score']:.3f} mean quality in `selector_summary_full_df`, while `phase_aware_aggressive` fails with {full_aggressive['mean_quality_score']:.3f}. This is the main reason the aggressive controller should be kept as an ablation instead of the final answer.

Among the fixed baselines on the common subset, `static_speculative` is the fastest quality-safe fixed option at {full_spec['mean_total_latency_ms']:.1f} ms in `selector_summary_common_subset_df`, while `static_int8` has lower mean prefill at {full_int8['mean_prefill_ms']:.1f} ms but a worse mean total latency of {full_int8['mean_total_latency_ms']:.1f} ms. That matches the idea that a simple controller should use speculative for decode-heavy cases and INT8 as the safer general branch.
"""
print(interpretation_text)


## Serving-Oriented Extension

The serving-oriented row stays separate on purpose.

It is a meaningful result, but it is not part of the core interactive selector comparison. The job of this section is just to show what the repo measurements say for that serving-style case.


In [ ]:
serving_extension_routes_df = build_selector_routes(
    workload_signals_df[workload_signals_df["workload_name"] == "ctx2048_out64"], ["phase_aware_quality"]
)
serving_extension_df = (
    stress_agg[stress_agg["workload_name"] == "ctx2048_out64"]
    .rename(
        columns={
            "ttft_ms_mean": "ttft_ms",
            "ttft_ms_std": "ttft_ms_std",
            "prefill_ms_mean": "prefill_ms",
            "prefill_ms_std": "prefill_ms_std",
            "tbt_ms_mean": "tbt_ms",
            "tbt_ms_std": "tbt_ms_std",
            "total_latency_ms_mean": "total_latency_ms",
            "total_latency_ms_std": "total_latency_ms_std",
            "tokens_per_second_mean": "tokens_per_second",
            "tokens_per_second_std": "tokens_per_second_std",
            "energy_per_token_j_mean": "energy_per_token_j",
            "energy_per_token_j_std": "energy_per_token_j_std",
            "peak_gpu_memory_mb_mean": "peak_gpu_memory_mb",
            "peak_gpu_memory_mb_std": "peak_gpu_memory_mb_std",
        }
    )[
        [
            "mode_name",
            "ttft_ms",
            "prefill_ms",
            "tbt_ms",
            "total_latency_ms",
            "tokens_per_second",
            "energy_per_token_j",
            "peak_gpu_memory_mb",
        ]
    ]
    .sort_values("total_latency_ms")
    .reset_index(drop=True)
)
serving_extension_df


## Final Conclusion

The meaningful final claim is pretty narrow and that is okay.

- The measurements do justify a lightweight selector.
- The cleanest evidence is not that every hybrid helps everywhere.
- The strongest phase-aware gain is the repeated-prefix branch.
- The conservative final choice is `phase_aware_quality`, not `phase_aware_aggressive`.
- The serving-style batching result is real, but it belongs in the serving extension section rather than in the default interactive controller.

So the selector story that survives a strict reading is:

- use simple runtime signals
- keep the controller small
- treat repeated-prefix routing as the clearest phase-aware win
- keep the aggressive GPTQ-heavy route as an ablation, not the final policy
